#  **Preprocessing**

In [0]:
!pip install kagglehub[pandas-datasets]

In [0]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "customer_support_tickets_200k.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "mirzayasirabdullah07/customer-support-tickets-dataset-200k-records",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

In [0]:
# fill na values
df['browser'] = df['browser'].fillna("No browser")
df

In [0]:
import re

# text normalization 
def normalize_text(text):
    """
    Standardizes text for the agent to read better
    """
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text
df['issue_description'] = df['issue_description'].apply(normalize_text)
df['resolution_notes'] = df['resolution_notes'].apply(normalize_text)

In [0]:
import pandas as pd

# columns that should be categorized
cat_cols = ['category', 'priority', 'status', 'escalated']

# create a new column with int values for each category 
# for each column listed above
for col in cat_cols:
    numerical_values, _ = pd.factorize(df[col])
    current_index = df.columns.get_loc(col)
    new_col_name = f'{col}_id'
    df.insert(current_index + 1, column = new_col_name, value = numerical_values)

# convert date columns to datetime
df['ticket_created_date'] = pd.to_datetime(df['ticket_created_date'])
df['ticket_resolved_date'] = pd.to_datetime(df['ticket_resolved_date'])

In [0]:
# Calculate resolution time in days
df['resolution_time_days'] = (df['ticket_resolved_date'] - df['ticket_created_date']).dt.days


In [0]:
pip install sentence-transformers

In [0]:
# create embeddings with batch processing
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

texts = df['issue_description'].tolist()

# Process in batches with progress bar
batch_size = 1000
embeddings = model.encode(texts, batch_size=batch_size, show_progress_bar=True)

df['embeddings'] = list(embeddings)
print(f"Created embeddings with shape: {embeddings.shape}")

# Exploratory Data **Analysis**

### Dataset Overview

In [0]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [0]:
#column types
df.info()

In [0]:
#summary stats
df.describe()

### Feature Analysis

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

categorical_cols = ['category', 'priority', 'status', 'escalated']

for col in categorical_cols:
    plt.figure(figsize=(8,4))
    sns.countplot(data=df, x=col, order=df[col].value_counts().index)
    plt.title(f"Distribution of {col}")
    plt.xticks(rotation=45)
    plt.show()


### Text analysis

In [0]:
%pip install nltk

In [0]:
from collections import Counter
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

# Combine all issue descriptions
all_text = " ".join(df['issue_description'])

# Tokenize
words = [w for w in all_text.split() if w not in stop_words]

# Most common words
word_freq = Counter(words).most_common(20)
word_freq


In [0]:
word_df = pd.DataFrame(word_freq, columns=['word', 'count'])
plt.figure(figsize=(10,5))
plt.bar(word_df['word'], word_df['count'])
plt.xticks(rotation=45)
plt.title("Top 20 Most Common Words in Issue Descriptions")
plt.show()


### Embedding

In [0]:
from sklearn.decomposition import PCA
import numpy as np

# Convert embeddings column to array
emb_matrix = np.vstack(df['embeddings'].values)

# Reduce to 2D
pca = PCA(n_components=2)
emb_2d = pca.fit_transform(emb_matrix)

# Plot
plt.figure(figsize=(10,7))
plt.scatter(emb_2d[:,0], emb_2d[:,1], s=5, alpha=0.5)
plt.title("PCA Visualization of Ticket Embeddings")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()


In [0]:
#grouping by meaning 
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import numpy as np

emb_matrix = np.vstack(df['embeddings'].values)
pca = PCA(n_components=10).fit_transform(emb_matrix)

# Cluster embeddings
kmeans = KMeans(n_clusters=5, random_state=42)
df['cluster'] = kmeans.fit_predict(pca)

# Visualize clusters
plt.figure(figsize=(10,7))
sns.scatterplot(x=pca[:,0], y=pca[:,1], hue=df['cluster'], palette='tab10', s=10)
plt.title("Ticket Embedding Clusters")
plt.show()

In [0]:
from collections import Counter

def top_words_for_cluster(cluster_id, n=20):
    texts = df[df['cluster'] == cluster_id]['issue_description']
    all_words = " ".join(texts).split()
    common = Counter(all_words).most_common(n)
    return common

for c in sorted(df['cluster'].unique()):
    print(f"\nCluster {c} top words:")
    print(top_words_for_cluster(c))


In [0]:
def sample_tickets(cluster_id, n=5):
    return df[df['cluster'] == cluster_id]['issue_description'].head(n)

for c in sorted(df['cluster'].unique()):
    print(f"\nCluster {c} sample tickets:")
    print(sample_tickets(c))

In [0]:
#escalation rate by cluster
cluster_escalation = (
    df.groupby('cluster')
      .apply(lambda x: (x['escalated'] == 'Yes').mean())
      .sort_values()
)

cluster_escalation

In [0]:
cluster_resolution = df.groupby('cluster')['resolution_time_days'].mean().sort_values()
cluster_resolution
